In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

2026-02-10 21:14:53.554004: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770758093.847139      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770758093.930040      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770758094.605566      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770758094.605626      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770758094.605629      17 computation_placer.cc:177] computation placer alr

In [2]:
lines = pd.read_csv('/kaggle/input/next-word-prediction/1661-0.txt',sep="\t",names=["data"],header=None)
texts = lines.data.tolist()
print(f"Lines extracted: {len(texts)}")

Lines extracted: 9633


In [3]:
tokens_count = sum([len(line) for line in texts])
print(f"Total tokens count: {tokens_count}")

Total tokens count: 569577


In [4]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
word_index_count = len(tokenizer.word_index)
print(f"Unique word count: {word_index_count}")

Unique word count: 8930


In [5]:
import random
random.sample(list(tokenizer.word_index.items()), 5)

[('he', 11),
 ('theirs', 6312),
 ('flecked', 8543),
 ('derives', 8703),
 ('“eh', 4361)]

In [6]:
tokenizer.texts_to_sequences(['under the terms of the project'])[0]

[262, 1, 480, 4, 1, 145]

In [7]:
input_sequences = []
for text in texts:
    sequence = tokenizer.texts_to_sequences([text])[0]
    for i in range(1,len(sequence)):
        input_sequences.append(sequence[:i+1])
len(input_sequences)

101619

In [8]:
max_len = max([len(seq) for seq in input_sequences])
print(f"Maximum length of sequence: {max_len}")

Maximum length of sequence: 20


In [9]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_seq = pad_sequences(input_sequences, maxlen = max_len,padding='pre')
padded_seq

array([[   0,    0,    0, ...,    0,  145, 4789],
       [   0,    0,    0, ...,  145, 4789,    1],
       [   0,    0,    0, ..., 4789,    1, 1020],
       ...,
       [   0,    0,    0, ...,    3,  360,   83],
       [   0,    0,    0, ...,  360,   83,  358],
       [   0,    0,    0, ...,   83,  358, 1673]], dtype=int32)

In [10]:
X, y = padded_seq[:,:-1], padded_seq[:,-1]

In [11]:
from tensorflow.keras.utils import to_categorical
y = to_categorical(y,num_classes=word_index_count+1)

In [12]:
X.shape, y.shape

((101619, 19), (101619, 8931))

In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [14]:
model = Sequential()
model.add(Embedding(word_index_count+1, 100, input_length=max_len-1))
model.add(LSTM(150, return_sequences=True))
model.add(LSTM(150))
model.add(Dense(word_index_count+1, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-02-10 21:15:13.374849: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [15]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [16]:
model.fit(X,y,epochs=10)

Epoch 1/10
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 191s 59ms/step - accuracy: 0.0547 - loss: 6.6559
Epoch 2/10
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 187s 59ms/step - accuracy: 0.0966 - loss: 5.7644
Epoch 3/10
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 186s 59ms/step - accuracy: 0.1235 - loss: 5.4132
Epoch 4/10
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 187s 59ms/step - accuracy: 0.1386 - loss: 5.1543
Epoch 5/10
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 188s 59ms/step - accuracy: 0.1499 - loss: 4.9311
Epoch 6/10
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 188s 59ms/step - accuracy: 0.1607 - loss: 4.7334
Epoch 7/10
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 188s 59ms/step - accuracy: 0.1692 - loss: 4.5632
Epoch 8/10
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 188s 59ms/step - accuracy: 0.1756 - loss: 4.4004
Epoch 9/10
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 189s 59ms/step - accuracy: 0.1839 - loss: 4.2354
Epoch 10/10
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 188s 59ms/step - accuracy: 0.1944 - loss: 4.0786


In [17]:
test_data = "as i passed"

In [18]:
for i in range(10):
    token_seq = tokenizer.texts_to_sequences([test_data])[0]
    padded_test_seq = pad_sequences([token_seq],maxlen = max_len,padding='pre')
    pos = np.argmax(model.predict(padded_test_seq))
    for word,index in tokenizer.word_index.items():
        if index == pos:
            test_data = test_data + " " + word
            print(test_data)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 367ms/step
as i passed down
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
as i passed down the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
as i passed down the door
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
as i passed down the door and
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
as i passed down the door and the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
as i passed down the door and the other
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
as i passed down the door and the other was
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
as i passed down the door and the other was a
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
as i passed down the door and the other was a very
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
as i passed down the door and the other was a very man
